# UZIMA Fabric Data Access

This notebook reads the ODBC connection string from Key Vault.

The connection string uses the `cdiofabric` managed identity to connect to Fabric. It must not use browser or email sign-in.

Run this from an Azure resource that has the `cdiofabric` user-assigned managed identity attached.

In [ ]:
from azure.identity import ManagedIdentityCredential
from azure.keyvault.secrets import SecretClient
import pandas as pd
import pyodbc

# Change only this value when you need another database.
# Options: "uzima_db_backup", "HCW_fitbit_data", "Qualtrics"
database_name = "uzima_db_backup"
managed_identity_client_id = "4ae6ed7b-b72c-4853-9a3c-10699e60f63e"

vault = SecretClient(
    vault_url="https://uzima-fabric-tokens.vault.azure.net/",
    credential=ManagedIdentityCredential(client_id=managed_identity_client_id),
)

connection_string = vault.get_secret("fabric-odbc-connection-string").value

if "Interactive" in connection_string:
    raise SystemExit("Key Vault ODBC secret must use managed identity auth, not browser or email sign-in.")

if "Authentication=ActiveDirectoryMsi" not in connection_string:
    raise SystemExit("Key Vault ODBC secret is missing Authentication=ActiveDirectoryMsi.")

parts = connection_string.split(";")
parts = [f"DATABASE={database_name}" if part.upper().startswith("DATABASE=") else part for part in parts]
connection_string = ";".join(parts)

connection = pyodbc.connect(connection_string, timeout=30)

## See Available Tables

In [ ]:
tables = pd.read_sql(
    """
    SELECT TABLE_SCHEMA, TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    ORDER BY TABLE_SCHEMA, TABLE_NAME
    """,
    connection,
)

tables.head(30)

## Read A Small Sample

After you see the table list, change `table_to_read` to the approved table or masked view you need.

In [ ]:
table_to_read = "dbo.dimenrolledparticipants"

sample = pd.read_sql(f"SELECT TOP 10 * FROM {table_to_read}", connection)
sample

## Combine Two Approved Tables

Use this only when `cdiofabric` has access to both approved tables or masked views.

In [ ]:
merge_question = """
SELECT TOP 100
  p.Gender,
  p.Age,
  f.date,
  f.steps
FROM uzima_db_backup.dbo.dimenrolledparticipants p
JOIN HCW_fitbit_data.dbo.fitbitdailydata f
  ON p.ParticipantIdentifier = f.participantidentifier
"""

# Remove the # on the next two lines when you need this combined dataset.
# merged_data = pd.read_sql(merge_question, connection)
# merged_data

In [ ]:
connection.close()